In [65]:
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold

from noise_score_eval import (
    compare_filtering_curves,
    evaluate_all_noise_scores,
    summarize_filtering_curve,
)

rng = np.random.default_rng(33)
X, y_clean = make_classification(
    n_samples=180,
    n_features=8,
    n_informative=5,
    n_redundant=0,
    n_classes=3,
    class_sep=1.0,
    random_state=33,
)

noise_idx = rng.choice(len(y_clean), size=36, replace=False)
y_noisy = y_clean.copy()
labels = np.unique(y_clean)
for idx in noise_idx:
    candidates = labels[labels != y_noisy[idx]]
    y_noisy[idx] = rng.choice(candidates)

noisy_ground_truth = np.zeros(len(y_clean), dtype=bool)
noisy_ground_truth[noise_idx] = True

good_score = noisy_ground_truth.astype(float) + rng.normal(scale=0.1, size=len(y_clean))
good_score[5] = np.nan
random_score = rng.random(len(y_clean))
constant_score = np.full(len(y_clean), 0.5)
noise_scores = {
    'good': good_score,
    'random': random_score,
    'constant': constant_score,
}

ranking_df = evaluate_all_noise_scores(noisy_ground_truth, noise_scores)
assert list(ranking_df.columns) == ['method', 'roc_auc', 'average_precision', 'precision_at_k', 'recall_at_k', 'f1_at_k']
assert ranking_df['roc_auc'].notna().all()
assert np.isclose(
    ranking_df.loc[ranking_df['method'] == 'constant', 'roc_auc'].iloc[0],
    0.5,
)
display(ranking_df)


,method,roc_auc,average_precision,precision_at_k,recall_at_k,f1_at_k
0,good,1.000000,1.000000,1.000000,1.000000,1.000000
1,random,0.494792,0.202928,0.194444,0.194444,0.194444
2,constant,0.500000,0.200000,0.222222,0.222222,0.222222


In [67]:
curve_df = compare_filtering_curves(
    X,
    y_noisy,
    noise_scores,
    LogisticRegression(max_iter=500),
    balanced_accuracy_score,
    removal_percentages=[0, 10, 20, 30],
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=33),
    random_state=33,
)
summary_df = summarize_filtering_curve(curve_df)

assert set(curve_df.columns) == {'method', 'removal_percentage', 'n_removed_mean', 'n_train_remaining_mean', 'metric_mean', 'metric_std'}
assert set(summary_df.columns) == {'method', 'best_metric', 'best_removal_percentage', 'mean_metric_across_percentages', 'auc_metric_curve', 'metric_at_0_removal', 'max_delta_vs_no_filter'}

display(curve_df)
display(summary_df)


AttributeError: module 'numpy' has no attribute 'trapz'